In [1]:
import numpy as np
import torch
import time
from transformers import Trainer, TrainingArguments

from evaluation.metrics_trainer_callback import SaveMetricsCallback
from evaluation.metrics import compute_ece, compute_B_std, compute_B_mean
from data_loading.get_datasets import get_glue_dataset
from models.get_roberta import get_hypernet_on_last_layer_roberta

/home/lewy700/Documents/roberta-hypernet-custom/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

glue_dataset_name = "cola"
model_name = "roberta-base"
lora_r = 1
lora_alpha = 16
hypernet_hidden_dim = 16

print(f"Hypernet on: {glue_dataset_name}")

Hypernet on: cola


In [3]:
model, tokenizer, hypernet = get_hypernet_on_last_layer_roberta(model_name=model_name, lora_r=lora_r, lora_alpha=lora_alpha, hypernet_hidden_dim=hypernet_hidden_dim)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
encoded_dataset, metric = get_glue_dataset(glue_dataset_name, tokenizer, truncation=True, max_length=512)

In [5]:
def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

        predictions = np.argmax(probs, axis=-1)
        results = metric.compute(predictions=predictions, references=labels)

        results["ece"] = compute_ece(probs, labels)
        results["hyper_B_std"] = compute_B_std(hypernet, device=device)
        results["hyper_B_mean"] = compute_B_mean(hypernet, device=device)

        return results

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./outputs/hypernet_{glue_dataset_name}",
    eval_strategy="epoch",
    # eval_steps=5,
    save_strategy="steps",
    save_steps=1000000000,
    learning_rate=4e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    num_train_epochs=20, # 80
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    weight_decay=0.1,
    disable_tqdm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[SaveMetricsCallback(f"./results", f"hypernet_{glue_dataset_name}_{str(int(time.time()))}.csv")]
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# def grad_hook_h(grad):
#     print(f"[BACKWARD] grad shape: {grad.shape}, mean: {grad.abs().mean().item():.8f} HYPERNET")
#     return grad

# def grad_hook(grad):
#     print(f"[BACKWARD] grad shape: {grad.shape}, mean: {grad.abs().mean().item():.8f}")
#     return grad

# hypernet.fc1.weight.register_hook(grad_hook_h)
# model.roberta.encoder.layer[-2].attention.self.query.lora_A["default"].weight.register_hook(grad_hook)

In [8]:
trainer.train()

tensor(0, device='cuda:0')
tensor([-0.0219,  0.1220, -0.8962,  0.4237,  0.1923, -0.2559, -2.0017,  0.7494],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)
tensor(1, device='cuda:0')
tensor([-0.5467, -2.1533, -0.1700,  0.6521,  0.2748,  0.0918, -0.3373,  1.1707],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)
[BACKWARD] grad shape: torch.Size([16, 776]), mean: 0.01017005 HYPERNET
[BACKWARD] grad shape: torch.Size([1, 768]), mean: 0.00000000
tensor(0, device='cuda:0')
tensor([-0.0219,  0.1220, -0.8962,  0.4237,  0.1923, -0.2559, -2.0017,  0.7494],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)
tensor(1, device='cuda:0')
tensor([-0.5467, -2.1533, -0.1700,  0.6521,  0.2748,  0.0918, -0.3373,  1.1707],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)
[BACKWARD] grad shape: torch.Size([16, 776]), mean: 0.00174407 HYPERNET
[BACKWARD] grad shape: torch.Size([1, 768]), mean: 0.00000000


Epoch,Training Loss,Validation Loss,Matthews Correlation,Ece,Hyper B Std,Hyper B Mean
1,No log,0.728370,-0.032090,0.122968,0.276257,-0.000267
2,0.710200,0.632448,0.001454,0.045039,0.548900,0.000638
3,0.710200,1.394857,0.000000,0.297025,0.384059,0.013251


tensor(0, device='cuda:0')
tensor([-0.0219,  0.1220, -0.8962,  0.4237,  0.1923, -0.2559, -2.0017,  0.7494],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)
tensor(1, device='cuda:0')
tensor([-0.5467, -2.1533, -0.1700,  0.6521,  0.2748,  0.0918, -0.3373,  1.1707],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)
[BACKWARD] grad shape: torch.Size([16, 776]), mean: 0.00057275 HYPERNET
[BACKWARD] grad shape: torch.Size([1, 768]), mean: 0.00000000
tensor(0, device='cuda:0')
tensor([-0.0219,  0.1220, -0.8962,  0.4237,  0.1923, -0.2559, -2.0017,  0.7494],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)
tensor(1, device='cuda:0')
tensor([-0.5467, -2.1533, -0.1700,  0.6521,  0.2748,  0.0918, -0.3373,  1.1707],
       device='cuda:0', grad_fn=<EmbeddingBackward0>)
[BACKWARD] grad shape: torch.Size([16, 776]), mean: 0.00507603 HYPERNET
[BACKWARD] grad shape: torch.Size([1, 768]), mean: 0.00000000
tensor(0, device='cuda:0')
tensor([-0.0218,  0.1220, -0.8962,  0.4237,  0.1923, -0.2

KeyboardInterrupt: 